In [1]:
# https://www.geeksforgeeks.org/machine-learning/frequent-pattern-growth-algorithm/
# https://medium.com/@anilcogalan/fp-growth-algorithm-how-to-analyze-user-behavior-and-outrank-your-competitors-c39af08879db

In [2]:
import pandas as pd
import tensorflow as tf
import pickle
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

Load and clean the data

In [3]:
df = pd.read_csv('grocery_baskets.csv')
print(df)

                                           item_name  basket_id
0                                            chicken          0
1                                               salt          0
2                                       acorn squash          0
3                                               sage          0
4                                           rosemary          0
...                                              ...        ...
143673                                     olive oil      13500
143674                                corn tortillas      13500
143675                                   black beans      13500
143676                                     pine nuts      13500
143677  a to quart shallow flameproof casserole dish      13500

[143678 rows x 2 columns]


In [4]:
# got error when trying to do the te not all items were strings so cleaning here
df["item_name"] = (
    df["item_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df = df[df["item_name"].notna() & (df["item_name"] != "") & (df["item_name"] != "nan")]
# all text, all lowercase, not extra whitespace, remove empty/invalid names

Convert baskets so each list == one basket

In [5]:
basket_lists = df.groupby("basket_id")["item_name"].apply(list).tolist()

Encode baskets for rule association

In [6]:
# https://rasbt.github.io/mlxtend/user_guide/preprocessing/TransactionEncoder/
te = TransactionEncoder()
basket_df = pd.DataFrame(
    te.fit_transform(basket_lists),
    columns=te.columns_
)


In [7]:
# drop any items that don't appear often
basket_df = basket_df.loc[:, basket_df.mean() > 0.01]

FP growth- find items that appear together and create the rules

In [8]:
frequent_items = fpgrowth(basket_df, min_support=0.01, use_colnames=True)

rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.3
)

rules = rules[
    (rules["lift"] > 1.2) &
    (rules["confidence"] > 0.3)
]

In [9]:
# Load task 1 model
model = tf.keras.models.load_model("grocery_classifier.keras")

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("label_encoder.pkl", "rb") as f:
    label_map = pickle.load(f)

Predict the items category ie. carrot > fruit and vegetables and recommend an extra item

In [10]:
def predict_category(item_name: str) -> str:
    input_tensor = tf.constant([item_name])
    probs = model.predict(input_tensor, verbose=0)
    label_index = int(tf.argmax(probs, axis=1)[0])
    return label_map[label_index]


def get_basket_categories(basket):
    return set(predict_category(item) for item in basket)

In [11]:
# Predict categories for all the items in a basket
def recommend_extra_items(basket, rules, top_n=3):
    basket = set(basket)
    basket_categories = get_basket_categories(basket)

    # Copy so the rules don't get changed
    rules = rules.copy()
    rules["overlap"] = rules["antecedents"].apply(
        lambda a: len(a & basket)
    )

    # Ranks the rule by any sort of relevance or confidence
    rules = rules.sort_values(
        by=["overlap", "lift", "confidence"],
        ascending=False
    )

    recommendations = []
    # Loops through recommended items
    for conseq in rules["consequents"]:
        item = list(conseq)[0]

        # Recommends items from categories in baskets and avoids recommending any duplicates
        if predict_category(item) not in basket_categories:
            continue

        if item not in basket and item not in recommendations:
            recommendations.append(item)

        if len(recommendations) == top_n:
            break

    return recommendations


Make command line recommender

In [17]:
def cli_recommender():
    print("Enter grocery items separated by commas (e.g. pasta, tomato, cheese)")
    print("Type 'quit' to exit")

    while True:
        user_input = input("Basket: ").strip()
        if user_input.lower() in {"quit", "exit"}:
            break

        basket = [item.strip().lower() for item in user_input.split(",")]

        recs = recommend_extra_items(basket, rules)

        if recs:
            print("Recommended extra items:")
            for r in recs:
                print(r)
        else:
            print("No suitable recommendations found.")

        print()

In [20]:
cli_recommender()
# highly inaccurate maybe instead of fp growth technique try another like apiori?
# accuracy did get a tiny bit better after a few changes to the dataset and model itself, so could try further messing with that?

Enter grocery items separated by commas (e.g. pasta, tomato, cheese)
Type 'quit' to exit
Recommended extra items:
ground cinnamon
salt
extra virgin olive oil



KeyboardInterrupt: 